# ORIE 5256 Team Exercises: Advances in Financial Machine Learning

Each team solves three of the five exercises below (2.1, 3.1, 7.2, 13.2, 14.2).

**Dependencies:** 3.1, 13.2 and 14.2 start from the dollar bars built in 2.1, so 2.1 should be one of the three. 7.2 uses the (X, y) dataset produced in 3.1, so 7.2 only makes sense together with 3.1.

**Submission:** one notebook per team. All reasoning and explanations go in code comments.

**Data:** E-mini S&P 500 (ES) futures tick data, 1997-2015 (`ES-csv.zip`, 1.9 GB zipped, ~34 GB unzipped). It will not fit in memory, so document the out-of-core approach used.

In [ ]:
# Setup and data-handling approach
#
# PROBLEM: ES.csv is 34 GB unzipped (1.9 GB zipped); it cannot be loaded into RAM.
# APPROACH: stream the CSV once out of the zip in fixed-size chunks and write each chunk
# to a year-partitioned, zstd-compressed Parquet dataset. Every later cell (bars, CUSUM,
# O-U fit, ...) reads Parquet, never the CSV.
#   - Streaming from the zip member avoids writing 34 GB of raw CSV to disk, and the
#     chunk size bounds peak RAM no matter how large the file is. We pipe `unzip -p`
#     because Python's zipfile cannot decompress this archive (unsupported method,
#     typical for >4 GB zips); the system unzip can.
#   - Parquet is columnar and typed: ~5x smaller than CSV, no re-parsing of text on every
#     read, and we can load only the columns we need.
#   - Partitioning by year lets later cells read one year at a time (or a slice for
#     development) instead of the whole history.
#   - data/ is gitignored, so none of this ever enters version control.
from __future__ import annotations  # lets `int | None` hints work on Python 3.9 kernels

import shutil
import subprocess
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Configuration over hardcoding: one place to change paths and sizes.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ZIP_PATH = REPO_ROOT / "ES-csv.zip"
CSV_NAME = "ES.csv"
PARQUET_DIR = REPO_ROOT / "data" / "interim" / "ticks_parquet"
CHUNK_ROWS = 5_000_000  # ~250 MB per chunk in pandas; small enough for any laptop
MAX_ROWS = None  # set e.g. 2_000_000 to smoke-test on a slice (writes to a separate dir)

# The file has no header. Raw time is HHMMSSmmm (e.g. 000649000 = 00:06:49.000) and is
# read as an integer, so we avoid slow string parsing on 34 GB of text.
COLUMNS = ["contract", "date", "time", "price", "size", "type"]
DTYPES = {"contract": "string", "date": "int32", "time": "int32",
          "price": "float64", "size": "int32", "type": "string"}
# float64 price is exact here: ES ticks are multiples of 0.25 (a power of two).


def to_timestamp(date: pd.Series, time: pd.Series) -> pd.Series:
    """Combine YYYYMMDD and HHMMSSmmm ints into one datetime64[ms] column."""
    day = pd.to_datetime(date.astype("int64").astype("string"), format="%Y%m%d")
    t = time.astype("int64")
    offset_ms = ((t // 10_000_000) * 3_600_000 + (t // 100_000 % 100) * 60_000
                 + (t // 1_000 % 100) * 1_000 + t % 1_000)
    return (day + pd.to_timedelta(offset_ms, unit="ms")).astype("datetime64[ms]")


def convert_csv_to_parquet(out_dir: Path, max_rows: int | None) -> None:
    """Stream ES.csv from the zip into year-partitioned Parquet, validating each chunk."""
    marker = out_dir / "_SUCCESS"
    # Idempotence: a finished conversion is never redone. A directory without the
    # marker is a crashed run, so it is wiped (partial parts would duplicate rows).
    if marker.exists():
        print(f"Parquet already built at {out_dir}; skipping conversion.")
        return
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True)

    n_rows, ts_min, ts_max = 0, None, None
    proc = subprocess.Popen(["unzip", "-p", str(ZIP_PATH), CSV_NAME], stdout=subprocess.PIPE)
    try:
        reader = pd.read_csv(proc.stdout, header=None, names=COLUMNS, dtype=DTYPES,
                             chunksize=CHUNK_ROWS, nrows=max_rows)
        for i, chunk in enumerate(reader):
            # Fail fast: a bad parse should stop the job now, not after hours of bars.
            assert not chunk.isna().any().any(), f"nulls in chunk {i}"
            chunk["timestamp"] = to_timestamp(chunk.pop("date"), chunk.pop("time"))
            chunk["year"] = chunk["timestamp"].dt.year.astype("int16")
            for year, part in chunk.groupby("year"):  # a chunk can straddle New Year
                year_dir = out_dir / f"year={year}"
                year_dir.mkdir(exist_ok=True)
                part = part.drop(columns="year")
                pq.write_table(pa.Table.from_pandas(part, preserve_index=False),
                               year_dir / f"part-{i:05d}.parquet", compression="zstd")
            n_rows += len(chunk)
            lo, hi = chunk["timestamp"].min(), chunk["timestamp"].max()
            ts_min = lo if ts_min is None else min(ts_min, lo)
            ts_max = hi if ts_max is None else max(ts_max, hi)
            print(f"chunk {i}: {n_rows:,} rows so far, up to {hi}", flush=True)
    finally:
        proc.stdout.close()  # unzip gets SIGPIPE if we stop early (MAX_ROWS)
        proc.wait()
    if proc.returncode not in (0, -13):  # -13 = SIGPIPE from early stop
        raise RuntimeError(f"unzip failed with code {proc.returncode}")
    marker.write_text(f"rows={n_rows}\nmin={ts_min}\nmax={ts_max}\n")  # last: marks success
    print(f"Done: {n_rows:,} rows, {ts_min} -> {ts_max}")


# Smoke-test runs go to their own directory so they never masquerade as the full dataset.
_out_dir = PARQUET_DIR if MAX_ROWS is None else PARQUET_DIR.with_name("ticks_parquet_sample")
convert_csv_to_parquet(_out_dir, MAX_ROWS)

# DOWNSTREAM USE: read one year (or a few columns) at a time, e.g.
#   pq.read_table(PARQUET_DIR, filters=[("year", "==", 2008)], columns=["timestamp", "price", "size"])
# Exercise 2.1 builds tick/volume/dollar bars year by year from this dataset.

## Exercise 2.1

On a series of E-mini S&P 500 futures tick data:
(a) Form tick, volume, and dollar bars. Use the ETF trick to deal with the roll.
(b) Count the number of bars produced by tick, volume, and dollar bars on a weekly basis. Plot a time series of that bar count. What bar type produces the most stable weekly count? Why?
(c) Compute the serial correlation of returns for the three bar types. What bar method has the lowest serial correlation?
(d) Partition the bar series into monthly subsets. Compute the variance of returns for every subset of every bar type. Compute the variance of those variances. What method exhibits the smallest variance of variances?
(e) Apply the Jarque-Bera normality test on returns from the three bar types. What method achieves the lowest test statistic?

In [ ]:
# Exercise 2.1


## Exercise 3.1

Form dollar bars for E-mini S&P 500 futures:
(a) Apply a symmetric CUSUM filter (Chapter 2, Section 2.5.2.1) where the threshold is the standard deviation of daily returns (Snippet 3.1).
(b) Use Snippet 3.4 on a pandas series t1, where numDays=1.
(c) On those sampled features, apply the triple-barrier method, where ptSl=[1,1] and t1 is the series you created in point 1.b.
(d) Apply getBins to generate the labels.

In [ ]:
# Exercise 3.1


## Exercise 7.2

Take a pair of matrices (X, y), representing observed features and labels. These could be one of the datasets derived from the exercises in Chapter 3.
(a) Derive the performance from a 10-fold CV of an RF classifier on (X, y), without shuffling.
(b) Derive the performance from a 10-fold CV of an RF on (X, y), with shuffling.
(c) Why are both results so different?
(d) How does shuffling leak information?

In [ ]:
# Exercise 7.2


## Exercise 13.2

Fit the time series of dollar bars of E-mini S&P 500 futures to an O-U process. Given those parameters:
(a) Produce a heat-map of Sharpe ratios for various profit-taking and stop-loss levels.
(b) What is the OTR?

In [ ]:
# Exercise 13.2


## Exercise 14.2

On the dollar bars dataset for E-mini S&P 500 futures, compute
(a) HHI index on positive returns.
(b) HHI index on negative returns.
(c) HHI index on time between bars.
(d) The 95-percentile DD.
(e) The 95-percentile TuW.
(f) Annualized average return.
(g) Average returns from hits (positive returns).
(h) Average return from misses (negative returns).
(i) Annualized SR.
(j) Information ratio, where the benchmark is the risk-free rate.
(k) PSR.
(l) DSR, where we assume there were 100 trials, and the variance of the trials' SR was 0.5.

In [ ]:
# Exercise 14.2
